In [67]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_recall_curve
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import os



In [37]:
#test_train_data = pd.read_csv("cleaned_data/test_train_data.csv")
#d3 = pd.read_csv("cleaned_data/d3.csv")
#modelling_data = pd.read_csv("cleaned_data/modelling_data.csv")
fulld1 = pd.read_csv("cleaned_data/fulld1.csv")

if os.path.exists('cleaned_data/d3_updated.csv'):
    d3 = pd.read_csv('cleaned_data/d3_updated.csv')
    print(f"Loaded d3_updated — anomaly count: {d3['is_anomaly'].sum()}")
else:
    d3 = pd.read_csv('cleaned_data/d3.csv')
    print(f"Loaded original d3 — anomaly count: {d3['is_anomaly'].sum()}")

Loaded d3_updated — anomaly count: 148


In [64]:
#creating my datasets for testing, training, modeling 

test_train_data = fulld1[
    fulld1['name'].isin(list(d3['name']))
]
exclude = test_train_data['name']

modelling_data = fulld1[~fulld1['name'].isin(exclude)]

#test_train_data['is_anomaly'] = test_train_data['is_anomaly'].fillna(0).astype(int)





test_train_data = test_train_data.merge(d3, on=['name', 'date', 'set'], how='left')
#test_train_data = test_train_data.merge(olympic, on=['name', 'date', 'set'], how='left')

test_train_data['is_anomaly'] = test_train_data[
    ['is_anomaly_x', 'is_anomaly_y']
].max(axis=1).fillna(0).astype(int)

test_train_data = test_train_data.drop(columns=['is_anomaly_x', 'is_anomaly_y'])


# test_train_data = test_train_data.drop(columns=['is_anomaly_x', 'is_anomaly_y'])


test_train_data

# print(len(good2))

# print(len(good))



# for i in good:
#     if i not in good2:
#         print(i)
        

print(test_train_data['is_anomaly'].sum())
#merged = olympic_keys.merge(df, on=['name', 'date', 'set'], how='left', indicator=True)


#n = test_train_data.loc[test_train_data['is_anomaly'] == 1, 'name']


# noe = []
# for i in n:
#     if i not in train_d:
#        noe.append(i)

#     elif i not in test_d:
#         noe.append(i)


# n

143


In [39]:
#train/test split 80/20 spit

d3_names = d3['name'].unique()
d3_only = test_train_data[test_train_data['name'].isin(d3_names)].copy()


unique_athletes = d3_only['name'].unique()

train_athletes, test_athletes = train_test_split(
    unique_athletes,
    test_size=0.2,
    random_state=42
)

train_df = d3_only[d3_only['name'].isin(train_athletes)]
test_df  = d3_only[d3_only['name'].isin(test_athletes)]

drop_cols = ['name', 'date', 'datetime', 'set','days_since_last_test']
train_df = train_df.drop(columns=drop_cols)
test_df  = test_df.drop(columns=drop_cols)

X_train = train_df.drop(columns=['is_anomaly'])
y_train = train_df['is_anomaly']
X_test  = test_df.drop(columns=['is_anomaly'])
y_test  = test_df['is_anomaly']

In [40]:
print(f"Train size: {len(X_train)} rows, {y_train.sum()} anomalies")
print(f"Test size:  {len(X_test)} rows, {y_test.sum()} anomalies")

train_names = set(train_df.index)
overlap = set(train_athletes) & set(test_athletes)
print(f"Athlete overlap between train and test: {len(overlap)}")

Train size: 4502 rows, 109 anomalies
Test size:  1282 rows, 34 anomalies
Athlete overlap between train and test: 0


In [41]:
train_set = set(train_athletes)
d3_set = set(d3['name'].unique())
overlap = train_set.intersection(d3_set)

print("Number of overlapping names(training):", len(overlap))
print(overlap)

Number of overlapping names(training): 96
{'andrew clarke', 'emmanuella frimpomaa', 'cameron davenport', 'konstantinos dellas', 'ian scott', 'aidan plummer', 'vasileios moiras', 'conor mcnally', 'luke lowery', 'lilly venezia', 'ava carlson', 'max livingston', 'jamal cooper', 'abby reger', 'mk daly', 'waleed qadir', 'emmett croteau', 'dylan baumgarth', "colin o'garro", 'brooke hess', 'grayson saunier', 'audrey marin', 'izzy mundee', 'jack mulholland', 'riley grosdidier', 'riley dumigan', 'marco dzamba', 'zyion brown', 'nick marinaro', 'john stenberg', 'george hawley', 'debra hill', 'tyler juhlin', 'godson ajoku', 'francisco caballero', 'isaiah golonka', 'john bancone', 'olivia lyman', 'samuel washington', 'panagiotis karagiorgis', 'kyle meier', 'tyson grimm', 'ahmir braxton', 'ava davis', 'chris miller', 'carson franks', 'caroline haggerty', 'patrick campbell', 'matthew scheible', 'mary lundregan', 'oskari vuorio', 'mcallister burke', 'elliott fee', 'emily garrard', 'thai brown', 'olivi

In [42]:
test_set = set(test_athletes)
d3_set = set(d3['name'].unique())
overlap = test_set.intersection(d3_set)

print("Number of overlapping names(testing):", len(overlap))
print(overlap)

Number of overlapping names(testing): 25
{'steve simpkins', 'luke armistead', 'hailey rorick', 'florentina terra', 'delby lemieux', 'gergely hudak', 'calum langmuir', 'hudson kohler', 'kellie sutton', 'yuseph mustafa', 'emma olausson', "grayson o'bara", 'sarah shelburne', 'joe onuwabhagbe', 'bruce williams', 'lila browne', 'chase engdahl', 'sonoma adams', 'niquis ratcliff', 'kenneth edwards', 'immanuel johnson', "no'koi maddox", 'andrew belles', 'abayomi babalola', 'kate riley'}


In [43]:
print(f"Train anomalies: {y_train.sum()} out of {len(y_train)} rows")
print(f"Test anomalies: {y_test.sum()} out of {len(y_test)} rows")

# Check the ratio is roughly similar
print(f"\nTrain anomaly rate: {y_train.sum()/len(y_train)*100:.2f}%")
print(f"Test anomaly rate: {y_test.sum()/len(y_test)*100:.2f}%")

Train anomalies: 109 out of 4502 rows
Test anomalies: 34 out of 1282 rows

Train anomaly rate: 2.42%
Test anomaly rate: 2.65%


In [44]:
#Pipe line functions for supervised models

def build_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

models = {
    'Logistic Regression': build_pipeline(
        LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    ),
    'Random Forest': build_pipeline(
        RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
    ),
     'Hist Gradient Boosting': build_pipeline(
        HistGradientBoostingClassifier(
            class_weight='balanced',
            random_state=42,
            max_iter=100
        )
    )
}




In [45]:
#Running perscision tests and confusion matrices on the 3 supervised models

all_results = {}
all_scores = {}
all_predictions = {}

for name, pipeline in models.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    y_score = pipeline.predict_proba(X_test)[:, 1]
    # Results
    print(classification_report(y_test, y_pred, target_names=['Clean', 'Anomaly']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


    # Store model outputs
    all_scores[name] = y_score
    all_predictions[name] = y_pred

    # Build ranked results for this model
    results = X_test.copy()
    results['true_label'] = y_test.values
    results['predicted_label'] = y_pred
    results['anomaly_score'] = y_score

    results = results.sort_values('anomaly_score', ascending=False)

    all_results[name] = results



  Logistic Regression
              precision    recall  f1-score   support

       Clean       0.99      0.79      0.88      1248
     Anomaly       0.08      0.71      0.15        34

    accuracy                           0.79      1282
   macro avg       0.54      0.75      0.51      1282
weighted avg       0.97      0.79      0.86      1282

Confusion Matrix:
[[983 265]
 [ 10  24]]

  Random Forest
              precision    recall  f1-score   support

       Clean       0.99      0.99      0.99      1248
     Anomaly       0.62      0.47      0.53        34

    accuracy                           0.98      1282
   macro avg       0.80      0.73      0.76      1282
weighted avg       0.98      0.98      0.98      1282

Confusion Matrix:
[[1238   10]
 [  18   16]]

  Hist Gradient Boosting
              precision    recall  f1-score   support

       Clean       0.99      0.97      0.98      1248
     Anomaly       0.43      0.76      0.55        34

    accuracy                  

In [46]:
all_results['Logistic Regression'].head(20)

,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day,true_label,predicted_label,anomaly_score
412,41.3,0.000000,201.315741,-200.2,-42.475095,-14.158365,-66.733333,1,1,1.000000
3199,34.4,0.000000,19.723696,-101.4,-32.883265,-4.697609,-14.485714,1,1,0.999383
970,146.4,5.342916,1643.451336,-60.9,-30.904250,-30.904250,-60.900000,1,1,0.999003
984,146.5,7.430512,1491.908397,-60.0,-31.286080,-31.286080,-60.000000,1,1,0.998939
934,148.7,8.169300,1827.360039,-56.1,-24.614110,-24.614110,-56.100000,1,1,0.998172
3105,135.9,26.725499,1327.914480,-55.9,-7.592386,-3.796193,-27.950000,0,1,0.991365
991,144.2,6.355754,1676.330401,-98.6,0.609459,NaN,NaN,0,1,0.991221
955,148.5,5.418149,1779.605687,-97.7,-2.452379,NaN,NaN,0,1,0.990156
974,147.6,3.869468,1540.450287,-95.8,-6.601301,NaN,NaN,0,1,0.989905
851,198.6,21.768159,1624.508459,-44.6,14.201622,14.201622,-44.600000,0,1,0.989391


In [71]:
#absolute coefficient importance (LR)
importances = pd.Series(
    abs(models['Logistic Regression']
        .named_steps['model']
        .coef_[0]),
    index=X_train.columns
).sort_values(ascending=False)

print(importances)

jump_height_cm                1.826593
bw_lb                         0.565593
bw_change                     0.505188
jump_height_change            0.192247
bw_change_per_day             0.191597
force_at_zero_vel             0.140754
jump_height_change_per_day    0.051576
dtype: float64


In [47]:
all_results['Random Forest'].head(20)

,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day,true_label,predicted_label,anomaly_score
2567,191.1,46.122478,2187.917889,-64.7,14.142708,2.020387,-9.242857,1,1,0.99
1307,190.7,50.845699,2458.172233,-81.3,20.227210,2.889601,-11.614286,1,1,0.96
1866,196.4,58.207985,2567.203095,-57.0,17.759687,5.919896,-19.000000,1,1,0.91
11,178.2,56.533229,2411.467858,-38.6,13.699105,6.849552,-19.300000,1,1,0.91
4307,200.0,60.318608,2668.721707,-48.6,17.136823,4.284206,-12.150000,0,1,0.88
450,183.0,69.582830,2900.917889,-66.4,23.014481,3.287783,-9.485714,1,1,0.83
5280,205.1,44.195937,3040.651093,-23.9,5.397235,1.799078,-7.966667,1,1,0.82
4278,145.0,61.010836,2325.455014,-50.3,11.273792,2.818448,-12.575000,1,1,0.82
5148,141.4,30.022061,1586.297322,-15.9,5.670742,0.810106,-2.271429,0,1,0.81
162,173.7,41.412475,1782.588571,-23.3,9.246992,1.320999,-3.328571,1,1,0.77


In [48]:
#feature importance (RF)

importances = pd.Series(
    models['Random Forest'].named_steps['model'].feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances)


bw_change                     0.379840
bw_change_per_day             0.214892
jump_height_change            0.131359
jump_height_change_per_day    0.079208
bw_lb                         0.071577
force_at_zero_vel             0.070924
jump_height_cm                0.052199
dtype: float64


In [49]:
# all_results['Hist Gradient Boosting'].head(20)
lr_low = all_results['Hist Gradient Boosting'][
    all_results['Hist Gradient Boosting']['predicted_label'] == 0
]

print(lr_low.head(40))

      bw_lb  jump_height_cm  force_at_zero_vel  bw_change  jump_height_change  \
2138  175.7       40.181394        1700.585185        0.7            4.200466   
841   185.7       28.876839        1720.788201      -13.6           -4.736341   
802   200.9       35.686611        2002.863231       17.9            6.269900   
2002  183.5       51.790725        2043.129870       -8.3            9.717309   
853   187.2       26.610450        1682.671647      -11.9           -0.021291   
3076  139.4       29.074392        1525.529125        7.7            2.615655   
930   130.8        3.849414        1389.917733      -14.7           -0.651249   
3104  191.8       34.317885        2120.582769       55.1            3.592687   
917   182.0       14.783405        1523.088126      -21.1          -24.483432   
779   205.0       38.995512        2366.412512      -13.4            5.837964   
992   183.6       33.997364        2186.003280       39.4           27.641611   
843   200.3       35.914051 

In [50]:
lr_low = all_results['Hist Gradient Boosting'][
    all_results['Hist Gradient Boosting']['predicted_label'] == 1
]

print(lr_low.head(40))

      bw_lb  jump_height_cm  force_at_zero_vel  bw_change  jump_height_change  \
2567  191.1       46.122478        2187.917889      -64.7           14.142708   
450   183.0       69.582830        2900.917889      -66.4           23.014481   
4278  145.0       61.010836        2325.455014      -50.3           11.273792   
11    178.2       56.533229        2411.467858      -38.6           13.699105   
1866  196.4       58.207985        2567.203095      -57.0           17.759687   
1307  190.7       50.845699        2458.172233      -81.3           20.227210   
851   198.6       21.768159        1624.508459      -44.6           14.201622   
4307  200.0       60.318608        2668.721707      -48.6           17.136823   
807   183.9       36.160521        1786.657895      -59.8           31.234036   
809   184.1       35.990745        1802.313953      -61.1            8.400374   
811   183.0       36.775369        1851.901979      -61.0           16.035756   
835   198.5       40.119471 

In [70]:
#permutation_importance (HGB)
perm = permutation_importance(
    models['Hist Gradient Boosting'],
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

importances = pd.Series(
    perm.importances_mean,
    index=X_train.columns
).sort_values(ascending=False)

print(importances)

bw_change                     0.038612
jump_height_cm                0.005616
bw_lb                         0.002106
force_at_zero_vel             0.001950
bw_change_per_day             0.001482
jump_height_change_per_day    0.000624
jump_height_change           -0.000390
dtype: float64


In [51]:
# isolation forest for 

print(f"\n{'='*50}")
print(f"  Isolation Forest")
print(f"{'='*50}")

contamination = y_train.sum() / len(y_train)
print(f"Contamination rate: {contamination:.4f}")

iso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100
    ))
])

iso_pipeline.fit(X_train)

y_pred_iso = iso_pipeline.predict(X_test)
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

iso_raw_score = -iso_pipeline.decision_function(X_test)

iso_score = (iso_raw_score - iso_raw_score.min()) / (iso_raw_score.max() - iso_raw_score.min())

print(classification_report(y_test, y_pred_iso, target_names=['Clean', 'Anomaly']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_iso))

iso_results = X_test.copy()
iso_results['true_label'] = y_test.values
iso_results['predicted_label'] = y_pred_iso
iso_results['anomaly_score'] = iso_score
iso_results = iso_results.sort_values('anomaly_score', ascending=False)

all_results['Isolation Forest'] = iso_results
all_scores['Isolation Forest'] = iso_score
all_predictions['Isolation Forest'] = y_pred_iso


  Isolation Forest
Contamination rate: 0.0242
              precision    recall  f1-score   support

       Clean       0.99      0.96      0.98      1248
     Anomaly       0.31      0.62      0.41        34

    accuracy                           0.95      1282
   macro avg       0.65      0.79      0.69      1282
weighted avg       0.97      0.95      0.96      1282

Confusion Matrix:
[[1201   47]
 [  13   21]]


In [52]:
all_results['Isolation Forest'].head(20)

,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day,true_label,predicted_label,anomaly_score
412,41.3,0.000000,201.315741,-200.2,-42.475095,-14.158365,-66.733333,1,1,1.000000
3199,34.4,0.000000,19.723696,-101.4,-32.883265,-4.697609,-14.485714,1,1,0.960119
867,243.9,7.623854,3186.821123,46.6,-19.848081,-19.848081,46.600000,0,1,0.943959
847,242.6,12.132906,3484.065863,41.9,-25.610332,-25.610332,41.900000,0,1,0.938048
984,146.5,7.430512,1491.908397,-60.0,-31.286080,-31.286080,-60.000000,1,1,0.929599
954,246.2,7.870528,3030.668979,37.6,-24.121800,-24.121800,37.600000,0,1,0.929563
970,146.4,5.342916,1643.451336,-60.9,-30.904250,-30.904250,-60.900000,1,1,0.928934
806,243.7,4.926485,2918.712717,42.9,-31.761333,-31.761333,42.900000,0,1,0.928231
934,148.7,8.169300,1827.360039,-56.1,-24.614110,-24.614110,-56.100000,1,1,0.926971
1866,196.4,58.207985,2567.203095,-57.0,17.759687,5.919896,-19.000000,1,1,0.902948


In [53]:
comparison = X_test.copy()
comparison['true_label'] = y_test.values

for name in all_scores:
    comparison[f'{name}_score'] = all_scores[name]
    comparison[f'{name}_pred'] = all_predictions[name]

comparison.sort_values('Hist Gradient Boosting_score', ascending=False).head(50)

,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day,true_label,Logistic Regression_score,Logistic Regression_pred,Random Forest_score,Random Forest_pred,Hist Gradient Boosting_score,Hist Gradient Boosting_pred,Isolation Forest_score,Isolation Forest_pred
2567,191.1,46.122478,2187.917889,-64.7,14.142708,2.020387,-9.242857,1,0.959540,1,0.99,1,0.999854,1,0.756871,1
450,183.0,69.582830,2900.917889,-66.4,23.014481,3.287783,-9.485714,1,0.961724,1,0.83,1,0.999845,1,0.895314,1
4278,145.0,61.010836,2325.455014,-50.3,11.273792,2.818448,-12.575000,1,0.960676,1,0.82,1,0.999835,1,0.863052,1
11,178.2,56.533229,2411.467858,-38.6,13.699105,6.849552,-19.300000,1,0.936501,1,0.91,1,0.999824,1,0.873952,1
1866,196.4,58.207985,2567.203095,-57.0,17.759687,5.919896,-19.000000,1,0.964918,1,0.91,1,0.999815,1,0.902948,1
1307,190.7,50.845699,2458.172233,-81.3,20.227210,2.889601,-11.614286,1,0.983020,1,0.96,1,0.999815,1,0.871974,1
851,198.6,21.768159,1624.508459,-44.6,14.201622,14.201622,-44.600000,0,0.989391,1,0.75,1,0.999799,1,0.896261,1
4307,200.0,60.318608,2668.721707,-48.6,17.136823,4.284206,-12.150000,0,0.917333,1,0.88,1,0.999767,1,0.871963,1
807,183.9,36.160521,1786.657895,-59.8,31.234036,NaN,NaN,0,0.922683,1,0.52,1,0.999755,1,0.540229,0
809,184.1,35.990745,1802.313953,-61.1,8.400374,NaN,NaN,0,0.926781,1,0.51,1,0.999755,1,0.414654,0


In [54]:
anom = test_train_data[
    test_train_data['is_anomaly'] == 0
]

cols = [
    'bw_lb', 
    'jump_height_cm', 
    'force_at_zero_vel', 
    'bw_change', 
    'jump_height_change', 
    'jump_height_change_per_day', 
    'bw_change_per_day', 
    
]

anom[cols].describe()

,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,jump_height_change,jump_height_change_per_day,bw_change_per_day
count,5639.000000,5641.000000,5641.000000,5517.000000,5521.000000,5143.000000,5139.000000
mean,206.515730,34.889492,2221.982684,0.778684,-14.868192,-6.157025,0.203276
std,50.302945,8.888346,574.531937,9.719598,836.925091,316.136471,3.223568
min,110.600000,1.839345,973.917733,-98.600000,-57098.325432,-19032.775144,-44.600000
25%,171.200000,28.516827,1740.448145,-1.400000,-1.504699,-0.213404,-0.209127
50%,200.500000,34.488020,2211.835753,0.100000,0.010197,-0.000430,0.014286
75%,239.350000,40.652274,2630.276908,1.600000,1.544649,0.210343,0.242857
max,390.500000,87.815260,4368.656015,201.500000,62.794824,20.779189,100.750000


In [55]:
modelling_data['bw_change_per_day'] = modelling_data['bw_change_per_day'].replace([np.inf, -np.inf], np.nan)

print(f"Athletes in prediction set: {modelling_data['name'].nunique()}")
print(f"Total rows: {len(modelling_data)}")


Athletes in prediction set: 714
Total rows: 17621


In [56]:
# prediction_meta = modelling_data[['name', 'date', 'set']].copy()
# # Step 1: Drop identifier columns from prediction data
# # Keep name and date so you can report back to S&C staff

# drop_cols = ['name', 'date', 'datetime', 'sport', 'set', 'days_since_last_test']
# pred_df = modelling_data.drop(columns=drop_cols, errors='ignore')

# # Step 2: Make sure columns match training exactly
# # Drop is_anomaly if it exists
# if 'is_anomaly' in pred_df.columns:
#     pred_df = pred_df.drop(columns=['is_anomaly'])

# # Confirm columns match
# print("Training columns:", X_train.columns.tolist())
# print("Prediction columns:", pred_df.columns.tolist())
# print("Match:", X_train.columns.tolist() == pred_df.columns.tolist())

# # Step 3: Get predictions and scores from all 3 supervised models
# # prediction_meta['LR_score']  = models['Logistic Regression'].predict_proba(pred_df)[:, 1]
# # prediction_meta['LR_pred']   = models['Logistic Regression'].predict(pred_df)

# prediction_meta['RF_score']  = models['Random Forest'].predict_proba(pred_df)[:, 1]
# prediction_meta['RF_pred']   = models['Random Forest'].predict(pred_df)

# prediction_meta['HGB_score'] = models['Hist Gradient Boosting'].predict_proba(pred_df)[:, 1]
# prediction_meta['HGB_pred']  = models['Hist Gradient Boosting'].predict(pred_df)

# prediction_meta['IF_score'] = models['Isolation Forest'].predict_proba(pred_df)[:, 1]
# prediction_meta['IF_pred']  = models['Isolation Forest'].predict(pred_df)

# # Step 4: Ensemble logic
# prediction_meta['models_agreed'] = (
#     prediction_meta['LR_pred'] +
#     prediction_meta['RF_pred'] +
#     prediction_meta['HGB_pred']
# )

# # High confidence = all 3 agree it's an anomaly
# prediction_meta['high_confidence'] = (
#     prediction_meta['models_agreed'] == 2
# ).astype(int)

# # Review needed = at least 2 of 3 agree
# prediction_meta['review_needed'] = (
#     prediction_meta['models_agreed'] >= 2
# ).astype(int)

# # Step 5: Summary
# print(f"\nTotal rows predicted: {len(prediction_meta)}")
# print(f"High confidence anomalies (all 3 agree): {prediction_meta['high_confidence'].sum()}")
# print(f"Review needed (2+ models agree): {prediction_meta['review_needed'].sum()}")

# # Step 6: Show high confidence flags first
# high_conf = prediction_meta[prediction_meta['high_confidence'] == 1].sort_values('HGB_score', ascending=False)
# print(f"\nHigh confidence anomalies:")
# print(high_conf[['name', 'date', 'set', 'HGB_score', 'RF_score']].to_string())

In [57]:
prediction_meta = modelling_data[['name', 'date', 'set', 'bw_change', 'jump_height_change', 'bw_change_per_day','days_since_last_test']].copy()

# Drop identifier columns
drop_cols = ['name', 'date', 'datetime', 'sport', 'set', 'days_since_last_test']
pred_df = modelling_data.drop(columns=drop_cols, errors='ignore')

# Drop is_anomaly if it exists
if 'is_anomaly' in pred_df.columns:
    pred_df = pred_df.drop(columns=['is_anomaly'])

# Replace infinity values
pred_df = pred_df.replace([np.inf, -np.inf], np.nan)

# Confirm columns match
print("Training columns:", X_train.columns.tolist())
print("Prediction columns:", pred_df.columns.tolist())
print("Match:", X_train.columns.tolist() == pred_df.columns.tolist())

# Random Forest and HGB — use predict_proba
prediction_meta['RF_score']  = models['Random Forest'].predict_proba(pred_df)[:, 1]
prediction_meta['RF_pred']   = models['Random Forest'].predict(pred_df)

prediction_meta['HGB_score'] = models['Hist Gradient Boosting'].predict_proba(pred_df)[:, 1]
prediction_meta['HGB_pred']  = models['Hist Gradient Boosting'].predict(pred_df)

# Isolation Forest — uses decision_function, not predict_proba
# Need to manually impute and scale since it wasn't in the pipeline
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

# Fit on training data then transform prediction data
X_train_iso = imputer.fit_transform(X_train)
X_pred_iso  = imputer.transform(pred_df)

X_train_iso = scaler.fit_transform(X_train_iso)
X_pred_iso  = scaler.transform(X_pred_iso)

# Fit Isolation Forest
iso = IsolationForest(
    contamination=y_train.sum() / len(y_train),
    random_state=42,
    n_estimators=100
)
iso.fit(X_train_iso)

# Get scores — more negative = more anomalous
# Normalize to 0-1 where higher = more anomalous
iso_raw_scores = iso.decision_function(X_pred_iso)
iso_scores_normalized = 1 - (
    (iso_raw_scores - iso_raw_scores.min()) / 
    (iso_raw_scores.max() - iso_raw_scores.min())
)

# Predict — convert -1 (anomaly) to 1, 1 (clean) to 0
iso_pred = np.where(iso.predict(X_pred_iso) == -1, 1, 0)

prediction_meta['ISO_score'] = iso_scores_normalized
prediction_meta['ISO_pred']  = iso_pred

# Ensemble — now using 3 models (RF, HGB, ISO)
prediction_meta['models_agreed'] = (
    prediction_meta['RF_pred'] +
    prediction_meta['HGB_pred'] +
    prediction_meta['ISO_pred']
)

# High confidence = all 3 agree
prediction_meta['high_confidence'] = (
    prediction_meta['models_agreed'] >= 3
).astype(int)

# Review needed = specifically HGB and ISO agree
prediction_meta['review_needed'] = (
    (prediction_meta['HGB_pred'] == 1) &
    (prediction_meta['ISO_pred'] == 1)
).astype(int)

# HGB only — fallback if no agreement between models
prediction_meta['hgb_only'] = (
    (prediction_meta['HGB_pred'] == 1) &
    (prediction_meta['ISO_pred'] == 0) &
    (prediction_meta['RF_pred'] == 0)
).astype(int)

# Summary
print(f"\nTotal rows predicted: {len(prediction_meta)}")
print(f"High confidence (all 3 agree): {prediction_meta['high_confidence'].sum()}")
print(f"Review needed (HGB + ISO agree): {prediction_meta['review_needed'].sum()}")
print(f"HGB only flags: {prediction_meta['hgb_only'].sum()}")

# Tiered if statement
high_conf = prediction_meta[
    prediction_meta['high_confidence'] == 1
].sort_values('HGB_score', ascending=False)

if len(high_conf) > 0:
    print(f"\nHigh confidence anomalies found: {len(high_conf)}")

else:
    high_conf = prediction_meta[
        prediction_meta['review_needed'] == 1
    ].sort_values('HGB_score', ascending=False)

    if len(high_conf) > 0:
        print(f"\nNo high confidence found — showing HGB + ISO agreement: {len(high_conf)}")

    else:
        high_conf = prediction_meta[
            prediction_meta['hgb_only'] == 1
        ].sort_values('HGB_score', ascending=False)
        print(f"\nNo model agreement — showing HGB only flags: {len(high_conf)}")

print(high_conf[['name', 'date', 'set', 'bw_change', 'jump_height_change',
                  'days_since_last_test', 'HGB_score', 'RF_score', 'ISO_score']].to_string())

Training columns: ['bw_lb', 'jump_height_cm', 'force_at_zero_vel', 'bw_change', 'jump_height_change', 'jump_height_change_per_day', 'bw_change_per_day']
Prediction columns: ['bw_lb', 'jump_height_cm', 'force_at_zero_vel', 'bw_change', 'jump_height_change', 'jump_height_change_per_day', 'bw_change_per_day']
Match: True

Total rows predicted: 17621
High confidence (all 3 agree): 1
Review needed (HGB + ISO agree): 1
HGB only flags: 49

High confidence anomalies found: 1
               name        date  set  bw_change  jump_height_change  days_since_last_test  HGB_score  RF_score  ISO_score
6264  eli goldstein  2025-05-28    1      -22.2           15.611345                  37.0    0.75252      0.53   0.794084


In [58]:
new_labels = high_conf[['name', 'date', 'set']].copy()
new_labels['is_anomaly'] = 1


d3_updated = pd.concat([d3, new_labels], ignore_index=True).drop_duplicates(
    subset=['name', 'date', 'set']
)


print(f"Original d3 anomalies: {d3['is_anomaly'].sum()}")
print(f"New anomalies added: {len(new_labels)}")
print(f"Updated d3 total: {d3_updated['is_anomaly'].sum()}")

Original d3 anomalies: 148
New anomalies added: 1
Updated d3 total: 149


In [59]:
# d3_updated.to_csv('cleaned_data/d3_updated.csv', index=False)
# print("d3_updated saved successfully")

In [63]:
full_assumed = fulld1.copy()

full_assumed = full_assumed.rename(columns={'is_anomaly': 'staff_indicated_anomaly'})

print(f"Total rows: {len(full_assumed)}")
print(f"Unique athletes: {full_assumed['name'].nunique()}")
print(f"Staff indicated anomalies: {full_assumed['staff_indicated_anomaly'].sum()}")

full_assumed['bw_change_per_day'] = full_assumed['bw_change_per_day'].replace(
    [np.inf, -np.inf], np.nan
)
full_assumed['jump_height_change_per_day'] = full_assumed['jump_height_change_per_day'].replace(
    [np.inf, -np.inf], np.nan
)

# Keep meta for reporting — include staff_indicated_anomaly
full_meta = full_assumed[['name', 'date', 'set', 'bw_lb',
                           'jump_height_cm', 'force_at_zero_vel',
                           'bw_change', 'bw_change_per_day',
                           'jump_height_change', 'jump_height_change_per_day',
                           'days_since_last_test',
                           'staff_indicated_anomaly']].copy()

drop_cols = ['name', 'date', 'datetime', 'sport', 'set',
             'days_since_last_test', 'staff_indicated_anomaly']
full_pred_df = full_assumed.drop(columns=drop_cols, errors='ignore')

full_pred_df = full_pred_df.replace([np.inf, -np.inf], np.nan)

print("\nMatch:", X_train.columns.tolist() == full_pred_df.columns.tolist())

full_meta['HGB_score'] = models['Hist Gradient Boosting'].predict_proba(full_pred_df)[:, 1]
full_meta['HGB_pred']  = models['Hist Gradient Boosting'].predict(full_pred_df)

imputer_full = SimpleImputer(strategy='median')
scaler_full  = StandardScaler()

X_train_iso_full = imputer_full.fit_transform(X_train)
X_pred_iso_full  = imputer_full.transform(full_pred_df)

X_train_iso_full = scaler_full.fit_transform(X_train_iso_full)
X_pred_iso_full  = scaler_full.transform(X_pred_iso_full)

iso_full = IsolationForest(
    contamination=y_train.sum() / len(y_train),
    random_state=42,
    n_estimators=100
)
iso_full.fit(X_train_iso_full)

iso_raw_full = iso_full.decision_function(X_pred_iso_full)
iso_scores_full = 1 - (
    (iso_raw_full - iso_raw_full.min()) /
    (iso_raw_full.max() - iso_raw_full.min())
)

full_meta['ISO_score'] = iso_scores_full
full_meta['ISO_pred']  = np.where(iso_full.predict(X_pred_iso_full) == -1, 1, 0)

full_meta['model_indicated_anomaly'] = (
    (full_meta['HGB_pred'] == 1) &
    (full_meta['ISO_pred'] == 1)
).astype(int)

print(f"\nTotal rows predicted: {len(full_meta)}")
print(f"Staff indicated anomalies: {full_meta['staff_indicated_anomaly'].sum()}")
print(f"Model indicated anomalies (HGB + ISO): {full_meta['model_indicated_anomaly'].sum()}")

indicated = full_meta[
    full_meta['model_indicated_anomaly'] == 1
].sort_values('HGB_score', ascending=False)

print(f"\nModel indicated anomalies:")
print(indicated[['name', 'date', 'bw_lb', 'jump_height_cm',
                  'bw_change', 'bw_change_per_day',
                  'jump_height_change', 'jump_height_change_per_day',
                  'days_since_last_test', 'HGB_score', 'ISO_score',
                  'HGB_pred', 'ISO_pred',
                  'staff_indicated_anomaly']].to_string())

full_meta.to_csv('cleaned_data/full_prediction_dataset', index=False)


#print("\nResults saved to cleaned_data/full_dataset_indicated_anomalies.csv")

Total rows: 23405
Unique athletes: 835
Staff indicated anomalies: 0

Match: True

Total rows predicted: 23405
Staff indicated anomalies: 0
Model indicated anomalies (HGB + ISO): 71

Model indicated anomalies:
                      name        date  bw_lb  jump_height_cm  bw_change  bw_change_per_day  jump_height_change  jump_height_change_per_day  days_since_last_test  HGB_score  ISO_score  HGB_pred  ISO_pred  staff_indicated_anomaly
19239    roland waguespack  2024-01-25  194.4       53.180178      -78.9         -11.271429           19.347495                    2.763928                   7.0   0.999856   0.847191         1         1                        0
20300          sean harmon  2024-01-25  178.7       56.457134      -76.5         -10.928571           20.670539                    2.952934                   7.0   0.999856   0.853197         1         1                        0
11734       joshua johnson  2024-05-08  215.1       48.713510      -38.4         -19.200000           13

In [62]:
final = pd.read_csv("cleaned_data/full_prediction_dataset")
final

,name,date,set,bw_lb,jump_height_cm,force_at_zero_vel,bw_change,bw_change_per_day,jump_height_change,jump_height_change_per_day,days_since_last_test,staff_indicated_anomaly,HGB_score,HGB_pred,ISO_score,ISO_pred,model_indicated_anomaly
0,aaron power,2024-01-23,1,162.1,41.969734,2016.445474,NaN,NaN,NaN,NaN,-1.0,0,0.681138,1,0.037401,0,0
1,abayomi babalola,2024-04-08,1,219.1,40.531239,2665.327931,NaN,NaN,NaN,NaN,-1.0,0,0.000047,0,0.034888,0,0
2,abayomi babalola,2024-04-10,1,219.4,42.815915,2757.147258,0.3,0.150000,2.284676,1.142338,2.0,0,0.000097,0,0.114652,0,0
3,abayomi babalola,2024-04-15,1,219.2,41.563591,2704.433521,-0.2,-0.040000,-1.252324,-0.250465,5.0,0,0.000017,0,0.057327,0,0
4,abayomi babalola,2024-04-17,1,223.5,40.162056,2682.577615,4.3,2.150000,-1.401535,-0.700767,2.0,0,0.000264,0,0.193275,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23400,zyion brown,2025-10-29,1,246.7,42.717731,3349.450791,-5.1,-1.700000,9.385441,3.128480,3.0,0,0.011625,0,0.519243,0,0
23401,zyion brown,2025-11-02,1,251.7,38.390091,3065.137218,5.0,1.250000,-4.327640,-1.081910,4.0,0,0.000109,0,0.256546,0,0
23402,zyion brown,2025-11-05,1,248.2,43.188406,3282.850191,-3.5,-1.166667,4.798315,1.599438,3.0,0,0.001132,0,0.363247,0,0
23403,zyion brown,2025-11-16,1,254.2,38.215138,3148.763451,6.0,0.545455,-4.973268,-0.452115,11.0,0,0.000093,0,0.192156,0,0
